In [2]:
import os
import re
import time

import pandas as pd
import requests

In [3]:
IN_VITRO = re.compile(
    r"microsom|hepatocyte|in vitro|s9 fraction|liver fraction",
    re.I
)

IN_VIVO = re.compile(
    r"in vivo|plasma clearance|administered|dosed|iv |intravenous|oral",
    re.I
)

SPECIES = [
    ("human", r"\bhuman\b|\bHLM\b"),
    ("rat", r"\brat\b|\bRLM\b"),
    ("mouse", r"\bmouse\b|\bmice\b"),
    ("dog", r"\bdog\b"),
    ("monkey", r"\bmonkey\b|cynomolgus"),
]


def classify(desc):
    d = desc or ""

    vitro = bool(IN_VITRO.search(d))
    vivo = bool(IN_VIVO.search(d))

    if vitro and not vivo:
        domain = "in_vitro"
    elif vivo and not vitro:
        domain = "in_vivo"
    elif vitro and vivo:
        domain = "ambiguous"
    else:
        domain = "unknown"

    species = next(
        (
            name
            for name, pattern in SPECIES
            if re.search(pattern, d, re.I)
        ),
        "unspecified"
    )

    return domain, species


In [4]:
def fetch_chembl(
    standard_type,
    page_size=1000,
    max_pages=200,
    timeout=180,
    sleep_sec=0.5,
):
    """
    Fetch activities of a given standard_type from the ChEMBL Activity API.

    Parameters
    ----------
    standard_type : str
        e.g. "CL", "Fu", "PPB"
    page_size : int
        Number of activities fetched per request
    max_pages : int
        Maximum number of pages
    timeout : int
        HTTP timeout (seconds)
    sleep_sec : float
        Wait time between API calls

    Returns
    -------
    list[dict]
    """

    rows = []
    offset = 0

    base_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

    for page in range(max_pages):

        url = (
            f"{base_url}"
            f"?standard_type={standard_type}"
            f"&limit={page_size}"
            f"&offset={offset}"
        )

        print(f"\nPage {page + 1}/{max_pages}")
        print(f"offset = {offset}")

        # retry
        response = None

        for attempt in range(3):
            try:
                response = requests.get(
                    url,
                    timeout=timeout
                )
                response.raise_for_status()

                data = response.json()
                break

            except Exception as e:
                print(
                    f"  Request failed "
                    f"(attempt {attempt + 1}/3): {e}"
                )

                if attempt < 2:
                    time.sleep(3)
                else:
                    print("  Giving up.")
                    return rows

        activities = data.get("activities", [])

        if not activities:
            print("No more activities.")
            break

        for a in activities:
            rows.append({
                "smiles": a.get("canonical_smiles"),
                "relation": a.get("standard_relation"),  # '=', '>', '<' etc. (to identify censored values)
                "value": a.get("standard_value"),
                "units": a.get("standard_units"),
                "assay_id": a.get("assay_chembl_id"),
                "doc_id": a.get("document_chembl_id"),
                "desc": a.get("assay_description"),
                "activity_id": a.get("activity_id"),
            })

        offset += page_size

        total = data.get("page_meta", {}).get(
            "total_count",
            len(rows)
        )

        print(
            f"  downloaded: "
            f"{min(offset, total):,}/{total:,}"
        )

        if offset >= total:
            break

        time.sleep(sleep_sec)

    return rows


In [9]:
def run_chembl_download(
    standard_type,
    output_dir=r"../data/in_vitro_data",
    page_size=1000,
    max_pages=200,
):
    """
    Download ChEMBL ADMET endpoints and
    classify them as in-vitro / in-vivo / ambiguous / unknown.
    """

    print("=" * 70)
    print(f"Downloading ChEMBL standard_type = {standard_type}")
    print("=" * 70)

    # 1. Fetch
    rows = fetch_chembl(
        standard_type=standard_type,
        page_size=page_size,
        max_pages=max_pages,
    )

    print(f"\nRaw activities downloaded: {len(rows):,}")

    # 2. DataFrame
    df = pd.DataFrame(rows)

    if df.empty:
        print("No data returned.")
        return df

    # 3. Remove missing essential values
    df = df.dropna(
        subset=["smiles", "value"]
    ).copy()

    # 4. Domain / species classification
    df[["domain", "species"]] = (
        df["desc"]
        .apply(
            lambda d: pd.Series(
                classify(d)
            )
        )
    )

    # 5. Output directory
    os.makedirs(output_dir, exist_ok=True)

    filename = (
        f"chembl_{standard_type.replace('/', '_')}_raw.csv"
    )

    output_path = os.path.join(
        output_dir,
        filename
    )

    # 6. Save
    df.to_csv(
        output_path,
        index=False
    )

    # 7. Summary
    print("\n" + "=" * 70)
    print("RESULT")
    print("=" * 70)

    print(
        f"Saved {len(df):,} activities"
    )

    print(
        f"Output: {output_path}"
    )

    print("\n== domain x species (unique compounds) ==")

    group = (
        df.groupby(
            ["domain", "species"]
        )["smiles"]
        .nunique()
        .sort_values(
            ascending=False
        )
    )

    print(group.head(20).to_string())

    print("\n== units ==")

    print(
        df["units"]
        .value_counts()
        .head(10)
        .to_string()
    )

    print("\n== domain counts ==")

    print(
        df["domain"]
        .value_counts()
        .to_string()
    )

    return df


In [10]:
df_cl = run_chembl_download(
    standard_type="CL"
)
df_fu = run_chembl_download(
    standard_type="Fu"
)
df_t12 = run_chembl_download(
    standard_type="T1/2"  # ChEMBL writes T1/2 (T1_2 returns 0 records)
)


Page 1/200
offset = 0
  downloaded: 1,000/82,022

Page 2/200
offset = 1000
  downloaded: 2,000/82,022

Page 3/200
offset = 2000
  downloaded: 3,000/82,022

Page 4/200
offset = 3000
  downloaded: 4,000/82,022

Page 5/200
offset = 4000
  downloaded: 5,000/82,022

Page 6/200
offset = 5000
  downloaded: 6,000/82,022

Page 7/200
offset = 6000
  downloaded: 7,000/82,022

Page 8/200
offset = 7000
  downloaded: 8,000/82,022

Page 9/200
offset = 8000
  downloaded: 9,000/82,022

Page 10/200
offset = 9000
  downloaded: 10,000/82,022

Page 11/200
offset = 10000
  downloaded: 11,000/82,022

Page 12/200
offset = 11000
  downloaded: 12,000/82,022

Page 13/200
offset = 12000
  downloaded: 13,000/82,022

Page 14/200
offset = 13000
  downloaded: 14,000/82,022

Page 15/200
offset = 14000
  downloaded: 15,000/82,022

Page 16/200
offset = 15000
  downloaded: 16,000/82,022

Page 17/200
offset = 16000
  downloaded: 17,000/82,022

Page 18/200
offset = 17000
  downloaded: 18,000/82,022

Page 19/200
offset = 1

In [13]:
df_ppb = run_chembl_download(
    standard_type="PPB"
)


Page 1/200
offset = 0
  downloaded: 1,000/14,720

Page 2/200
offset = 1000
  downloaded: 2,000/14,720

Page 3/200
offset = 2000
  downloaded: 3,000/14,720

Page 4/200
offset = 3000
  downloaded: 4,000/14,720

Page 5/200
offset = 4000
  downloaded: 5,000/14,720

Page 6/200
offset = 5000
  downloaded: 6,000/14,720

Page 7/200
offset = 6000
  downloaded: 7,000/14,720

Page 8/200
offset = 7000
  downloaded: 8,000/14,720

Page 9/200
offset = 8000
  downloaded: 9,000/14,720

Page 10/200
offset = 9000
  downloaded: 10,000/14,720

Page 11/200
offset = 10000
  downloaded: 11,000/14,720

Page 12/200
offset = 11000
  downloaded: 12,000/14,720

Page 13/200
offset = 12000
  downloaded: 13,000/14,720

Page 14/200
offset = 13000
  downloaded: 14,000/14,720

Page 15/200
offset = 14000
  downloaded: 14,720/14,720

Raw activities downloaded: 14,720

RESULT
Saved 13,778 activities
Output: ..\data\in_vitro_data\chembl_PPB_raw.csv

== domain x species (unique compounds) ==
domain    species    
unknown   

In [11]:
# Check assay description samples per domain

pd.set_option(
    "display.max_colwidth",
    300
)

display(
    df_cl[
        ["domain", "species", "value", "units", "desc"]
    ]
    .sample(
        min(30, len(df_cl)),
        random_state=42
    )
)


AttributeError: 'Index' object has no attribute '_format_flat'

         domain      species    value          units  \
45490  in_vitro          rat    560.0   mL.min-1.g-1   
36881   unknown        mouse     15.5  mL.min-1.kg-1   
17622   unknown          rat  2139.45  mL.min-1.kg-1   
7355    unknown          rat     18.0  mL.min-1.kg-1   
34659  in_vitro          rat     0.01  mL.min-1.kg-1   
2033    unknown          rat     55.7  mL.min-1.kg-1   
5261    unknown          dog      1.0  mL.min-1.kg-1   
55025  in_vitro        human      6.3  mL.min-1.kg-1   
81732  in_vitro        human     4.02   mL.min-1.g-1   
22051  in_vitro        human     54.0   mL.min-1.g-1   
25858  in_vitro        human      8.0   mL.min-1.g-1   
79035  in_vitro        human     21.0  mL.min-1.kg-1   
38704  in_vitro        human     15.0   mL.min-1.g-1   
9192    unknown        human      8.9  mL.min-1.kg-1   
5957   in_vitro        human     13.0   mL.min-1.g-1   
60840   in_vivo          dog      5.0  mL.min-1.kg-1   
31991  in_vitro        human   131.83   mL.min-1

In [12]:
display(
    df_cl[
        df_cl["domain"].isin(
            ["ambiguous", "unknown"]
        )
    ][
        ["species", "value", "units", "desc"]
    ]
    .head(50)
)


AttributeError: 'Index' object has no attribute '_format_flat'

         species  value          units  \
6            rat  48.33  mL.min-1.kg-1   
11           dog   46.0  mL.min-1.kg-1   
12           dog   20.0  mL.min-1.kg-1   
13           dog   19.0  mL.min-1.kg-1   
19   unspecified   36.5  mL.min-1.kg-1   
31           dog  134.0  mL.min-1.kg-1   
32           dog  197.0  mL.min-1.kg-1   
38           dog   97.0  mL.min-1.kg-1   
39           dog   94.0  mL.min-1.kg-1   
47           dog   28.0  mL.min-1.kg-1   
48           dog   40.0  mL.min-1.kg-1   
49           rat   13.5  mL.min-1.kg-1   
50           rat  28.33  mL.min-1.kg-1   
51           dog  161.0  mL.min-1.kg-1   
52           dog  220.0  mL.min-1.kg-1   
53           dog  176.0  mL.min-1.kg-1   
59           rat   54.5  mL.min-1.kg-1   
63           dog   53.0  mL.min-1.kg-1   
95           rat   84.0  mL.min-1.kg-1   
111          rat   49.0  mL.min-1.kg-1   
192  unspecified  33.33  mL.min-1.kg-1   
198  unspecified  33.33  mL.min-1.kg-1   
200          dog   79.0  mL.min-1.